In [ ]:
#| default_exp llm


# LLM

In [ ]:
#| export
import json
import requests
import litellm
from litellm import completion

from llm_sandbox_ui.app_config import MODEL, KERNEL_URL, NOTEBOOK_SYS_PROMPT, UE_TOOL_SYS_PROMPT 
from llm_sandbox_ui.llm_tools import TOOLS, TOOL_SCHEMAS

litellm.drop_params = True


In [ ]:
#| export

class AgenticToolLoop():
    """Agentic loop that calls LLM with tools until completion or max iterations.
    
    Manages message history, tool execution, and streaming responses for
    multi-turn conversations with tool calling capabilities.
    
    Attributes:
        model: LiteLLM model identifier string.
        sys_prompt: System prompt for the conversation.
        local: If True, use local tools; else call Unreal Engine server.
        max_iters: Maximum tool call iterations before stopping.
        tools: List of tool schemas for the LLM.
        messages: Conversation history.
        llm_turn_result: Result dict from the last LLM turn.
    """

    def __init__(self,model, sys_prompt, local=True, max_iters = 30):
        """Initialize the agentic tool loop.
        
        Args:
            model: LiteLLM model identifier string.
            sys_prompt: System prompt for the conversation.
            local: If True, use local tools; else call Unreal Engine server.
            max_iters: Maximum tool call iterations before stopping.
        """
        self.model = model
        self.sys_prompt = sys_prompt
        self.local = local
        self.max_iters = max_iters

        self.tools = self.get_tools(local=local)
        self.messages = []

        self.llm_turn_result = None

    def get_tools(self, local = True):
        """Fetch available tool schemas.
        
        Args:
            local: If True, return local TOOL_SCHEMAS; else fetch from Unreal.
            
        Returns:
            List of tool schema dicts.
        """
        # Get Tools from Globals
        if local:
            global TOOL_SCHEMAS
            return TOOL_SCHEMAS
        # Or Get Tools From Unreal
        else:
            response = requests.get(f'{KERNEL_URL}/tools')
            return response.json()

    def run_single_tool(self, raw_tool_args, tool_name, local=True):
        """Execute a single tool call.
        
        Args:
            raw_tool_args: JSON string of tool arguments.
            tool_name: Name of the tool to execute.
            local: If True, run locally; else POST to Unreal server.
            
        Returns:
            String result from tool execution or error message.
        """
        args = json.loads(raw_tool_args)

        if local:
            # Run Local Tool
            try:
                result = TOOLS[tool_name](**args)
                return str(result)
            except Exception as e:
                return f"Local tool error: {str(e)}"
        else:
            # Send Request to Unreal Server
            response = requests.post(
                f'{KERNEL_URL}/execute_tool',
                json={'function': tool_name, 'arguments': args},
                timeout = 10)
            
            data = response.json()
            # Check if there's an error
            if 'error' in data:
                return f"Error: {data['error']}"
            
            # Check if result exists
            if 'result' not in data:
                return f"Unexpected response: {data}"
            return data['result']

    def run_tool_calls(self,tool_calls):
        """Execute a batch of tool calls and update message history.
        
        Args:
            tool_calls: List of dicts with 'tool_id', 'func_name', 'args' keys.
            
        Yields:
            Status strings for each tool execution.
        """
        for tool_call in tool_calls:

            # Get Tool Info
            tool_id = tool_call['tool_id']
            tool_name = tool_call['func_name']
            raw_tool_args = tool_call['args']

            # Execute Tool
            yield f'\n\n🔧 Executing Tool: {tool_name}({tool_call['args']})\n\n'

            # Leaving the possibility of inter-loop switching, local versus Unreal
            tool_result = self.run_single_tool(raw_tool_args,
                                               tool_name, 
                                               local=self.local) 
            
            # Append Tool History
            self.messages.append({'role': 'tool',
                                 'tool_call_id': tool_id,
                                 'content': json.dumps(tool_result)})

    def llm_turn(self):
        """Run one LLM completion turn with streaming.
        
        Accumulates text and tool calls from the stream, updates message
        history, and sets llm_turn_result with finish_reason and tool_calls.
        
        Yields:
            Text content chunks from the LLM response.
        """
        llm_stream = completion(
                                model=self.model,
                                messages = self.messages,
                                tools = self.tools,
                                stream=True,
                                )

        text_output = ""
        tool_name = None
        tool_call = ""
        tool_id = None
        tool_index = 0
        tool_calls = []

        for chunk in llm_stream:
            finish_reason = chunk.choices[0].finish_reason
            delta = chunk.choices[0].delta

            # Tool Arg Accumulation
            if delta.tool_calls:
                if delta.tool_calls[0].function.name:

                    # If Multiple Tool Calls, Stack them Up
                    if tool_index != delta.tool_calls[0].index:
                        tool_calls.append({'func_name':tool_name,
                                            'args':tool_call,
                                            'tool_id':tool_id})

                        tool_call = ""
                        tool_index = delta.tool_calls[0].index

                    tool_name = delta.tool_calls[0].function.name
                    tool_id = delta.tool_calls[0].id

                json_piece = delta.tool_calls[0].function.arguments
                tool_call += json_piece

            # Text Message Accumulation
            elif delta.content:
                yield delta.content
                text_output += delta.content

            # Finish 
            elif finish_reason:
                if text_output:
                    self.messages.append({'role':'assistant',
                                        'content':text_output})  

                if finish_reason == 'tool_calls':

                    tool_calls.append({'func_name':tool_name,
                                        'args':tool_call,
                                        'tool_id':tool_id})

                    # Build Message History from Tool Call List
                    tool_message_list = []
                    for tool_call in tool_calls:
                        tool_message_list.append({'id':tool_call['tool_id'],
                                                'type':'function',
                                                'function':{'name':tool_call['func_name'],
                                                            'arguments':tool_call['args']}})

                    self.messages.append({'role':'assistant',
                                        'tool_calls':tool_message_list})

                self.llm_turn_result = {'finish_reason':finish_reason,
                        'tool_calls':tool_calls}

    def call_llm(self, notebook_history, query):
        """Run the full agentic loop until completion.
        
        Args:
            notebook_history: List of prior conversation messages.
            query: User's query string.
            
        Yields:
            Text chunks and tool execution status messages.
        """
        # Build History Including Notebook
        self.messages=[
                {"role": "system", "content":self.sys_prompt},
                *notebook_history,
                {"role": "user", 'content':query}
                ]

        # Loop until Finish Reason says Stop
        finish_reason = 'go'
        iters = 0
        while finish_reason != 'stop':
            if iters > self.max_iters:
                yield '\nReached Max Tool Calls\n'
                break
            for chunk in self.llm_turn():
                yield chunk

            result = self.llm_turn_result
            finish_reason = result['finish_reason']

            # Execute Tool Calls
            if finish_reason == 'tool_calls':
                for chunk in self.run_tool_calls(result['tool_calls']):
                    yield chunk
            iters += 1

